In [3]:
import psycopg2
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
logger = logging.getLogger(__name__)

# ==============================
# KONEKSI POSTGRESQL
# ==============================
conn = psycopg2.connect(
    host="postgres",
    database="olist_ecommerce",
    user="airflow",
    password="airflow",
    port="5432"
)
cur = conn.cursor()
logger.info("✅ Koneksi PostgreSQL berhasil!")

try:
    # ==============================
    # BUAT SCHEMA STAGING
    # ==============================
    cur.execute("CREATE SCHEMA IF NOT EXISTS staging;")
    conn.commit()
    logger.info("✅ Schema staging berhasil dibuat!")

    # ==============================
    # BUAT TABEL STAGING
    # ==============================
    cur.execute("""
        CREATE TABLE IF NOT EXISTS staging.orders (
            order_id                        VARCHAR PRIMARY KEY,
            customer_id                     VARCHAR,
            order_status                    VARCHAR,
            order_purchase_timestamp        TIMESTAMP,
            order_approved_at               TIMESTAMP,
            order_delivered_carrier_date    TIMESTAMP,
            order_delivered_customer_date   TIMESTAMP,
            order_estimated_delivery_date   TIMESTAMP,
            created_at                      TIMESTAMP DEFAULT NOW()
        );

        CREATE TABLE IF NOT EXISTS staging.order_items (
            order_id        VARCHAR,
            order_item_id   INTEGER,
            product_id      VARCHAR,
            seller_id       VARCHAR,
            price           FLOAT,
            freight_value   FLOAT,
            created_at      TIMESTAMP DEFAULT NOW(),
            PRIMARY KEY (order_id, order_item_id)
        );

        CREATE TABLE IF NOT EXISTS staging.payments (
            order_id                VARCHAR,
            payment_sequential      INTEGER,
            payment_type            VARCHAR,
            payment_installments    INTEGER,
            payment_value           FLOAT,
            created_at              TIMESTAMP DEFAULT NOW(),
            PRIMARY KEY (order_id, payment_sequential)
        );

        CREATE TABLE IF NOT EXISTS staging.customers (
            customer_id         VARCHAR PRIMARY KEY,
            customer_unique_id  VARCHAR,
            customer_zip_code   VARCHAR,
            customer_city       VARCHAR,
            customer_state      VARCHAR,
            created_at          TIMESTAMP DEFAULT NOW()
        );

        CREATE TABLE IF NOT EXISTS staging.products (
            product_id                  VARCHAR PRIMARY KEY,
            product_category_name       VARCHAR,
            product_name_length         FLOAT,
            product_description_length  FLOAT,
            product_photos_qty          FLOAT,
            product_weight_g            FLOAT,
            product_length_cm           FLOAT,
            product_height_cm           FLOAT,
            product_width_cm            FLOAT,
            created_at                  TIMESTAMP DEFAULT NOW()
        );

        CREATE TABLE IF NOT EXISTS staging.sellers (
            seller_id        VARCHAR PRIMARY KEY,
            seller_zip_code  VARCHAR,
            seller_city      VARCHAR,
            seller_state     VARCHAR,
            created_at       TIMESTAMP DEFAULT NOW()
        );
    """)
    conn.commit()
    logger.info("✅ Tabel staging berhasil dibuat!")

    # ==============================
    # COPY RAW to STAGING
    # ==============================

    # Orders
    cur.execute("""
        INSERT INTO staging.orders
        SELECT * FROM raw.raw_orders
        ON CONFLICT (order_id) DO NOTHING;
    """)
    conn.commit()
    logger.info("✅ staging.orders loaded!")

    # Order Items
    cur.execute("""
        INSERT INTO staging.order_items
        SELECT * FROM raw.raw_order_items
        ON CONFLICT (order_id, order_item_id) DO NOTHING;
    """)
    conn.commit()
    logger.info("✅ staging.order_items loaded!")

    # Payments
    cur.execute("""
        INSERT INTO staging.payments
        SELECT * FROM raw.raw_payments
        ON CONFLICT (order_id, payment_sequential) DO NOTHING;
    """)
    conn.commit()
    logger.info("✅ staging.payments loaded!")

    # Customers
    cur.execute("""
        INSERT INTO staging.customers
        SELECT * FROM raw.customers
        ON CONFLICT (customer_id) DO NOTHING;
    """)
    conn.commit()
    logger.info("✅ staging.customers loaded!")

    # Products
    cur.execute("""
        INSERT INTO staging.products
        SELECT * FROM raw.products
        ON CONFLICT (product_id) DO NOTHING;
    """)
    conn.commit()
    logger.info("✅ staging.products loaded!")

    # Sellers
    cur.execute("""
        INSERT INTO staging.sellers
        SELECT * FROM raw.sellers
        ON CONFLICT (seller_id) DO NOTHING;
    """)
    conn.commit()
    logger.info("✅ staging.sellers loaded!")

except Exception as e:
    conn.rollback()
    logger.error(f"❌ Pipeline error: {e}")
    raise

finally:
    cur.close()
    conn.close()
    logger.info("🏁 Semua data berhasil dipindah ke staging!")

2026-03-28 15:41:24,941 - ✅ Koneksi PostgreSQL berhasil!
2026-03-28 15:41:24,943 - ✅ Schema staging berhasil dibuat!
2026-03-28 15:41:24,974 - ✅ Tabel staging berhasil dibuat!
2026-03-28 15:41:25,945 - ✅ staging.orders loaded!
2026-03-28 15:41:25,949 - ❌ Pipeline error: relation "raw.raw_order_items" does not exist
LINE 3:         SELECT * FROM raw.raw_order_items
                              ^

2026-03-28 15:41:25,952 - 🏁 Semua data berhasil dipindah ke staging!


UndefinedTable: relation "raw.raw_order_items" does not exist
LINE 3:         SELECT * FROM raw.raw_order_items
                              ^
